# AQE and Broadcast join

Here are point-by-point notes from the video [Broadcast Joins & AQE (Adaptive Query Execution)](https://youtu.be/bRjVa7MgsBM?si=9KTw1R0fuCBk7Uq1) focusing on understanding AQE and Broadcast Join for data skew:

### Adaptive Query Execution (AQE) for Data Skew

*   **AQE (Adaptive Query Execution)** is a technique introduced in **Spark 3.0** to solve data skew.
*   Spark uses **runtime statistics** to select the most efficient query plan with AQE enabled.
    *   These statistics include the **size of the input dataset** (number of bytes read) and the **number of partitions**.
*   AQE provides three main tuning optimizations:
    *   **Tuning Shuffle Partitions:**
        *   If there are many shuffle partitions but only a few distinct keys, AQE can **coalesce** the partitions.
        *   For example, if there are 200 shuffle partitions but only 15 distinct keys after a join, AQE can reduce the number of partitions to 15, avoiding empty partitions.
        *   This **reduces the number of shuffle partitions**, leading to fewer tasks and potentially less resource usage.
    *   **Optimizing Joins:**
        *   AQE can **convert Sort Merge Joins to Broadcast Joins** at runtime if one side of the join is small enough.
        *   **Sort Merge Joins** are operation-heavy, involving **shuffling** (costly network data transfer), **sorting**, and then **merging**.
        *   **Broadcast Joins** do not involve a shuffle, making them more efficient when applicable.
    *   **Optimizing Skewed Joins:**
        *   This is the most relevant optimization for data skew.
        *   AQE can **split large, skewed partitions** in a Sort Merge Join into **smaller partitions**.
        *   This allows the skewed partition to be processed in parallel by more executors, mitigating the bottleneck.
*   **Enabling Skewed Join Optimization with AQE**:
    *   The property `spark.sql.adaptive.skewJoin.enabled` must be set to **`true`**.
    *   This requires the general AQE enablement property `spark.sql.adaptive.enabled` to also be set to **`true`**.
    *   When enabled, Spark dynamically handles skew in Sort Merge Joins by splitting skewed partitions.
*   **Example of AQE in Action:**
    *   The video demonstrates joining a `transaction` dataset with a skewed `customer ID` distribution and a `customer` dataset.
    *   Without AQE, a Sort Merge Join on the skewed key results in one partition taking significantly longer to process, as seen in the Spark UI event timeline.
    *   With AQE enabled, the Spark UI shows a different query plan with an "**aqe Shuffle read**" step.
    *   Instead of the initial 200 shuffle partitions, AQE reads the data in **fewer partitions** (e.g., four in the example).
    *   The skewed partition is effectively broken down, resulting in **more evenly distributed processing times** across tasks. The difference between the minimum and maximum processing time for tasks is reduced with AQE.
    *   The query plan without AQE shows a scan, a shuffle with 200 partitions for both datasets, and then sort and merge.
    *   The query plan with AQE shows a scan, an exchange with 200 partitions, followed by "aqe Shuffle read" which then operates on a smaller number of partitions.
*   **Important Note:** AQE might not always be the best option, and manual tuning might still be necessary in some cases.

### Broadcast Join for Data Skew

*   **Broadcast Join** is another technique to address data skew and can be more effective than Sort Merge Join in scenarios where **one of the tables being joined is significantly smaller than the other**.
*   **Internal Working of Sort Merge Join (Recap):**
    *   **Shuffle:** Data is redistributed across executors based on the join key. This is the **most expensive operation** due to network transfer.
    *   The partition to which a key is sent is determined by a hash of the key modulo the number of shuffle partitions (`hash(key) mod shuffle_partitions`). For joins on multiple keys, Spark hashes all the keys together.
    *   Due to data skew, all records with a frequent join key will end up in the same partition after the shuffle.
    *   **Sort:** Data within each partition is sorted based on the join key.
    *   **Merge:** Sorted partitions from both datasets are merged to find matching keys and produce the joined result.
*   **Internal Working of Broadcast Join:**
    *   One of the datasets (the smaller one) is **copied and sent to all executor nodes**.
    *   The join then happens locally on each executor between the local partition of the larger dataset and the entire broadcasted smaller dataset.
*   **Immunity to Skew:** Broadcast Joins are **immune to skewed input data** on the larger table because you have flexibility in how the larger table is partitioned initially.
    *   You can **repartition the larger dataset evenly** before the broadcast join.
    *   Since the smaller table is fully replicated on each executor, the join operation is distributed regardless of the key distribution in the original larger partitions.
*   **Example of Broadcast Join:**
    *   Using the same `transaction` and `customer` datasets, enabling the `autoBroadcastJoinThreshold` property (e.g., to 10MB) will trigger a Broadcast Join if the `customer` dataset's size is below this threshold.
    *   The video shows that Broadcast Join can be significantly faster (e.g., 3.2 seconds) compared to AQE in this specific scenario where one table is small.
*   **Key Requirement:** Broadcast Join is efficient when one table is small enough to be broadcasted across the cluster. The `spark.sql.autoBroadcastJoinThreshold` property controls the maximum size of a table that will be broadcast.
*   **Salting:** The video mentions **salting** as another important technique for solving data skew, but it is not discussed in detail in this excerpt.

These notes should provide a comprehensive understanding of AQE and Broadcast Join for data skew based on the provided transcript, suitable for learning and interview preparation. Remember to verify specific configuration properties and Spark versions with the official Spark documentation.

# Questions

Here are some hard MCQs on AQE and Broadcast Join for data skew, based on the provided transcript:

1.  **Adaptive Query Execution (AQE)**, as introduced in Spark 3.0, primarily aims to improve query performance by:
    *   Optimizing data storage formats at runtime.
    *   Dynamically adjusting the query plan based on **runtime statistics**.
    *   Predicting and pre-computing query results.
    *   Allowing users to manually fine-tune execution parameters during query execution.

2.  Which of the following is **NOT** one of the main tuning optimizations provided by **Adaptive Query Execution (AQE)** according to the video?
    *   Tuning Shuffle Partitions by coalescing them based on the number of distinct keys.
    *   Optimizing Joins by converting Sort Merge Joins to Broadcast Joins.
    *   Optimizing Skewed Joins by splitting large partitions in a Sort Merge Join.
    *   Optimizing data partitioning at the source level before query execution.

3.  When **AQE** tunes shuffle partitions by coalescing, it does so based on:
    *   The initial number of partitions configured.
    *   The size of the data in each partition before shuffling.
    *   The number of **distinct keys** in the data after a shuffle.
    *   The processing time of each individual partition.

4.  **AQE** can convert a **Sort Merge Join** to a **Broadcast Join** if:
    *   Both datasets being joined are large and distributed evenly.
    *   The join keys have a uniform distribution across both datasets.
    *   One of the datasets being joined is determined to be sufficiently **small** at runtime based on statistics.
    *   The Sort Merge Join is experiencing significant data skew.

5.  What is the most computationally expensive operation in a standard **Sort Merge Join** due to network overhead?
    *   The sorting of data within each partition.
    *   The merging of sorted data from different partitions.
    *   The **shuffling** of data across the network based on the join key.
    *   The initial reading of data from disk.

6.  To enable the optimization of skewed joins using **Adaptive Query Execution (AQE)**, which of the following Spark SQL properties **must** be set to `true`?
    *   `spark.sql.adaptive.enabled` only.
    *   `spark.sql.skewJoin.enabled` only.
    *   Both `spark.sql.adaptive.enabled` and `spark.sql.skewJoin.enabled`.
    *   `spark.sql.adaptive.enabled` and `spark.sql.adaptive.autoBroadcastJoinThreshold`.

7.  When **AQE** optimizes a skewed **Sort Merge Join**, it addresses the skew by:
    *   Replicating the smaller partitions to match the size of the skewed partition.
    *   Dropping records in the skewed partition to achieve a more even distribution.
    *   **Splitting the large, skewed partition** into multiple smaller partitions.
    *   Changing the join type from Sort Merge to a different algorithm.

8.  In a **Sort Merge Join**, the partition to which a record with a specific join key is sent during the shuffle phase is typically determined by:
    *   A random assignment based on the number of executors.
    *   The alphabetical order of the join key.
    *   The **hash of the join key modulo the number of shuffle partitions**.
    *   The size of the record itself.

9.  **Broadcast Join** is particularly effective in mitigating data skew when:
    *   Both joining datasets are very large.
    *   The join keys in the larger dataset are highly skewed.
    *   One of the joining datasets is **small enough to be efficiently copied** to all executor nodes.
    *   Both joining datasets are partitioned using the same distribution.

10. What is the primary reason why **Broadcast Join** is considered **immune to skewed input data** in the larger table?
    *   The larger table is automatically repartitioned by Spark during the broadcast.
    *   The smaller table, which contains the skewed data, is replicated.
    *   You have the flexibility to **partition the larger dataset evenly** before the join, and the smaller dataset is fully replicated.
    *   Broadcast Join only operates on a sample of the larger dataset.

11. The Spark SQL property `spark.sql.autoBroadcastJoinThreshold` controls:
    *   The minimum size of a table that can be broadcasted.
    *   Whether or not Broadcast Join is enabled in the Spark session.
    *   The **maximum size of a table** (in bytes) that will be considered for **automatic broadcast joining**.
    *   The number of executors that will receive a copy of the broadcasted table.

12. Compared to a **Sort Merge Join**, a **Broadcast Join** avoids which costly operation?
    *   The sorting of data within partitions.
    *   The merging of sorted data.
    *   The **shuffling** of data across the network.
    *   The reading of data from disk.

# Answers

Here are the correct answers to the MCQs with brief explanations:

1.  *   **Dynamically adjusting the query plan based on runtime statistics**. AQE utilizes runtime data characteristics to select a more efficient execution strategy.

2.  *   **Optimizing data partitioning at the source level before query execution**. The video focuses on optimizations performed by Spark during query execution, not on pre-query source-level partitioning.

3.  *   The number of **distinct keys** in the data after a shuffle. If there are many partitions but few unique keys, AQE can reduce the number of partitions.

4.  *   One of the datasets being joined is determined to be sufficiently **small** at runtime based on statistics. Broadcast Join is more efficient when one table can fit in the memory of each executor.

5.  *   The **shuffling** of data across the network based on the join key. Shuffling involves data transfer between executors, making it a costly operation.

6.  *   Both **`spark.sql.adaptive.enabled`** and **`spark.sql.skewJoin.enabled`**. Both of these properties must be set to `true` to enable the adaptive skew join optimization.

7.  *   **Splitting the large, skewed partition** into multiple smaller partitions. This allows the work to be distributed more evenly across executors.

8.  *   The **hash of the join key modulo the number of shuffle partitions**. This determines which partition each record will be sent to during the shuffle phase.

9.  *   One of the joining datasets is **small enough to be efficiently copied** to all executor nodes. This allows the join to happen locally on each executor without a shuffle.

10. *   You have the flexibility to **partition the larger dataset evenly** before the join, and the smaller dataset is fully replicated. Even if the original partitioning of the larger table was skewed, repartitioning can address this before the broadcast.

11. *   The **maximum size of a table** (in bytes) that will be considered for **automatic broadcast joining**. If a table's size is below this threshold, Spark may automatically use a Broadcast Join.

12. *   The **shuffling** of data across the network. In a Broadcast Join, the smaller table is sent to all executors, eliminating the need to shuffle the larger table.